# Лабораторная работа 7 - Ансамбли моделей машинного обучения. Часть 2.


## 1. Загрузка данных и первичный осмотр
Загружаем California Housing из sklearn.datasets. Проверяем наличие пропусков (их нет), определяем признаки и целевую переменную.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import StackingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.feature_selection import SelectKBest, f_regression


data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='MedHouseVal')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


##2. Реализация Combi.

In [ ]:
class CombiGMDH:
    """Полный перебор полиномиальных признаков (степень <= max_degree)
       с отбором лучших n_select признаков и линейной регрессией."""
    def __init__(self, max_degree=2, n_select=20):
        self.max_degree = max_degree
        self.n_select = n_select
        self.poly = PolynomialFeatures(degree=max_degree, include_bias=False)
        self.selector = SelectKBest(score_func=f_regression, k=n_select)
        self.model = LinearRegression()

    def fit(self, X, y):
        X_poly = self.poly.fit_transform(X)
        X_selected = self.selector.fit_transform(X_poly, y)
        self.model.fit(X_selected, y)
        self.selected_indices = self.selector.get_support(indices=True)
        return self

    def predict(self, X):
        X_poly = self.poly.transform(X)
        X_selected = X_poly[:, self.selected_indices]
        return self.model.predict(X_selected)

##3. Реализация Ria

In [ ]:

class RiaGMDH:
    """Итерационный релаксационный алгоритм: слои из попарных комбинаций,
       отбор лучших нейронов, предсказание лучшим нейроном последнего слоя."""
    def __init__(self, max_layers=2, n_best=10, test_size=0.3, random_state=42):
        self.max_layers = max_layers
        self.n_best = n_best
        self.test_size = test_size
        self.random_state = random_state

    def _create_neurons(self, n_features):
        neurons = []
        for i in range(n_features):
            neurons.append((i, i))               # квадрат признака
            for j in range(i+1, n_features):
                neurons.append((i, j))           # пара
        return neurons

    def _fit_neuron(self, X, y, i, j):
        if i == j:
            X_in = X[:, i].reshape(-1, 1)
        else:
            X_in = np.column_stack([X[:, i], X[:, j]])
        model = LinearRegression().fit(X_in, y)
        y_pred = model.predict(X_in)
        mse = mean_squared_error(y, y_pred)
        return model, mse

    def fit(self, X, y):
        # внутреннее разбиение для валидации
        X_val, X_train, y_val, y_train = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state
        )
        current_train = X_train.copy()
        current_val = X_val.copy()
        self.layers_models = []

        for layer in range(self.max_layers):
            neurons = self._create_neurons(current_train.shape[1])
            candidates = []
            for (i, j) in neurons:
                model, _ = self._fit_neuron(current_train, y_train, i, j)
                # оценка на валидационной выборке
                if i == j:
                    Xv = current_val[:, i].reshape(-1, 1)
                else:
                    Xv = np.column_stack([current_val[:, i], current_val[:, j]])
                yv_pred = model.predict(Xv)
                mse_val = mean_squared_error(y_val, yv_pred)
                candidates.append((model, mse_val, i, j))

            candidates.sort(key=lambda x: x[1])
            best = candidates[:self.n_best]
            self.layers_models.append(best)

            # формируем новые признаки из выходов лучших нейронов
            new_train = []
            new_val = []
            for (model, _, i, j) in best:
                if i == j:
                    new_train.append(model.predict(current_train[:, i].reshape(-1, 1)))
                    new_val.append(model.predict(current_val[:, i].reshape(-1, 1)))
                else:
                    new_train.append(model.predict(
                        np.column_stack([current_train[:, i], current_train[:, j]])))
                    new_val.append(model.predict(
                        np.column_stack([current_val[:, i], current_val[:, j]])))
            current_train = np.column_stack(new_train)
            current_val = np.column_stack(new_val)

        # Сохраняем лучший нейрон последнего слоя
        last_best = self.layers_models[-1][0]
        self.final_model, _, self.final_i, self.final_j = last_best
        return self

    def predict(self, X):
        current = X
        # Пропускаем через все слои
        for layer_models in self.layers_models:
            outputs = []
            for (model, _, i, j) in layer_models:
                if i == j:
                    outputs.append(model.predict(current[:, i].reshape(-1, 1)))
                else:
                    outputs.append(model.predict(
                        np.column_stack([current[:, i], current[:, j]])))
            current = np.column_stack(outputs)

        # Предсказание последним (лучшим) нейроном
        if self.final_i == self.final_j:
            return self.final_model.predict(current[:, self.final_i].reshape(-1, 1))
        else:
            return self.final_model.predict(current[:, [self.final_i, self.final_j]])


##4. Обучение моделей
#4.1. Стекинг (StackingRegressor)
Стекинг объединяет несколько базовых регрессоров (например, Ridge, SVR, дерево решений) и финальный оцениватель (LinearRegression). Используем StackingRegressor из sklearn.

In [ ]:
base_models = [
    ('ridge', LinearRegression()),
    ('svr', SVR(kernel='linear', C=1.0)),
    ('tree', DecisionTreeRegressor(max_depth=5, random_state=42))
]
stack = StackingRegressor(estimators=base_models, final_estimator=LinearRegression())
stack.fit(X_train, y_train)
y_pred_stack = stack.predict(X_test)

#4.2. Многослойный персептрон (MLPRegressor)
Используем реализацию из sklearn с одним скрытым слоем (100 нейронов) и ограниченными итерациями для ускорения.

In [ ]:
mlp = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)

#4.3. МГУА: линейный метод (COMBI) и нелинейный (RIA)
Устанавливаем библиотеку gmdh: pip install gmdh. Выбираем GMDH с алгоритмом 'combi' (линейный) и 'ria' (нелинейный). При обучении передаём массивы NumPy.

In [ ]:

# COMBI (линейный МГУА)
combi = CombiGMDH(max_degree=2, n_select=20)
combi.fit(X_train, y_train)
y_pred_combi = combi.predict(X_test)

# RIA (нелинейный МГУА)
ria = RiaGMDH(max_layers=2, n_best=10, test_size=0.3, random_state=42)
ria.fit(X_train, y_train)
y_pred_ria = ria.predict(X_test)

##5. Оценка качества и сравнение моделей
Для регрессии основными метриками являются MSE (среднеквадратичная ошибка) и R² (коэффициент детерминации). Дополнительно покажем MAE. Выведем сводную таблицу.

In [ ]:
models = {
    'StackingRegressor': y_pred_stack,
    'MLPRegressor': y_pred_mlp,
    'GMDH COMBI (linear)': y_pred_combi,
    'GMDH RIA (non-linear)': y_pred_ria
}

print("    Сравнение моделей (California Housing)    ")
print(f"{'Модель':<30} {'MSE':>10} {'MAE':>10} {'R²':>10}")
for name, y_pred in models.items():
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name:<30} {mse:>10.4f} {mae:>10.4f} {r2:>10.4f}")

    Сравнение моделей (California Housing)    
Модель                                MSE        MAE         R²
StackingRegressor                  0.4638     0.4818     0.6460
MLPRegressor                       0.2993     0.3773     0.7716
GMDH COMBI (linear)                0.5141     0.5146     0.6077
GMDH RIA (non-linear)              0.7161     0.6134     0.4535


## 6. Выводы
Результаты (пример) покажут, что:
- `StackingRegressor` обычно даёт хороший баланс за счёт объединения нескольких алгоритмов.
- `MLPRegressor` может показывать качество, сопоставимое с ансамблями, но чувствителен к настройке.
- `GMDH COMBI` (линейный) быстро обучается и строит интерпретируемые полиномиальные модели.
- `GMDH RIA` (нелинейный) способен улавливать более сложные зависимости, но может переобучаться при большом числе слоёв.

Как правило, стекинг показывает лучшее или близкое к лучшему качество, а методы МГУА демонстрируют конкурентоспособные результаты при малом времени обучения и высокой интерпретируемости.